# 03. Model Benchmarking, Tuning and Controlled Ablations

**Course:** URP4201 Research Methodology

Benchmarking majority baselines, Logistic Regression, Random Forest, and XGBoost on the 20% holdout test partition.

In [1]:
import sys, os
from pathlib import Path
PROJECT_ROOT = Path("..").resolve() if Path("..").resolve().name == "AquaSense AI" else Path(".").resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd

# Load model comparison results on holdout partition
comp_df = pd.read_csv("results/model_comparison.csv")
print("--- Model Benchmark Comparison (20% Holdout Test Partition) ---")
comp_df

--- Model Benchmark Comparison (20% Holdout Test Partition) ---


,model_name,accuracy,macro_f1,weighted_f1,macro_pr_auc,brier_score
0,Dummy (Majority),0.560,0.2393,0.4021,0.6667,0.8800
1,Logistic Regression,0.995,0.9943,0.9950,0.9960,0.0145
2,Random Forest,0.995,0.9943,0.9950,0.9984,0.0137
3,XGBoost,0.975,0.9662,0.9752,0.9845,0.0446


In [2]:
# Controlled Ablations log across feature sets, class imbalance, and calibration
exp_df = pd.read_csv("results/experiment_log.csv")
print("--- Controlled Ablations Log ---")
exp_df

--- Controlled Ablations Log ---


,experiment,model,model_name,accuracy,macro_f1,weighted_f1,macro_pr_auc,brier_score
0,model_comparison,Dummy (Majority),Dummy (Majority),0.560,0.2393,0.4021,0.6667,0.8800
1,model_comparison,Logistic Regression,Logistic Regression,0.995,0.9943,0.9950,0.9960,0.0145
2,model_comparison,Random Forest,Random Forest,0.995,0.9943,0.9950,0.9984,0.0137
3,model_comparison,XGBoost,XGBoost,0.975,0.9662,0.9752,0.9845,0.0446
4,ablation,Raw 8 Features,NaN,0.000,1.0000,0.0000,1.0000,0.0064
5,ablation,Domain Engineered 13 Features,NaN,0.000,0.9943,0.0000,0.9984,0.0122
6,ablation,Standard (Unweighted),NaN,0.000,0.9943,0.0000,0.9989,0.0132
7,ablation,Cost-Sensitive (Balanced Weights),NaN,0.000,0.9943,0.0000,0.9984,0.0122
8,ablation,Uncalibrated Random Forest,NaN,0.000,0.9943,0.0000,0.9984,0.0152
9,ablation,Platt Calibrated Random Forest,NaN,0.000,0.9943,0.0000,0.9984,0.0122


In [3]:
# Verified headline metrics from results/metrics.json
with open("results/metrics.json", "r", encoding="utf-8") as f:
    metrics = json.load(f)

print("Dataset Partition:", metrics["dataset_split"])
print("Model Architecture:", metrics["model_architecture"])
print("Headline Performance Scores:")
for k, v in metrics["headline_metrics"].items():
    print(f"  {k:20}: {v}")

Dataset Partition: Independent Holdout Test Split (20% Stratified, 200 samples)
Model Architecture: Calibrated Random Forest with Domain Features (4-Way Partition)
Headline Performance Scores:
  macro_f1            : 0.9943
  accuracy            : 0.995
  weighted_f1         : 0.995
  macro_pr_auc        : 0.9984
  brier_score         : 0.0122


In [4]:
# Per-Class Precision, Recall, F1-Score, and PR-AUC breakdown
pd.DataFrame(metrics["per_class_metrics"]).T

,precision,recall,f1_score,pr_auc
Good,0.9912,1.000,0.9956,0.9995
Moderate,1.0000,0.975,0.9873,0.9958
Polluted,1.0000,1.000,1.0000,1.0000


In [5]:
# Holdout Test Confusion Matrix
cm = metrics["confusion_matrix"]
pd.DataFrame(
    cm,
    index=["Actual Good", "Actual Moderate", "Actual Polluted"],
    columns=["Predicted Good", "Predicted Moderate", "Predicted Polluted"]
)

,Predicted Good,Predicted Moderate,Predicted Polluted
Actual Good,112,0,0
Actual Moderate,1,39,0
Actual Polluted,0,0,48
